# 📘 MedQCNN: From Fundamentals to Advanced
# A Complete Learning Guide

This notebook takes you from **zero** to a full understanding of how MedQCNN works — covering classical neural networks, quantum computing, and how we fuse them for medical diagnostics.

## Table of Contents

| Chapter | Topic | Level |
|---------|-------|-------|
| 1 | Neural Networks & CNNs | 🟢 Fundamental |
| 2 | Medical Image Classification | 🟢 Fundamental |
| 3 | Quantum Computing Basics | 🟡 Intermediate |
| 4 | Quantum Gates & Circuits | 🟡 Intermediate |
| 5 | The Hybrid Architecture | 🟡 Intermediate |
| 6 | Amplitude Encoding | 🔴 Advanced |
| 7 | Variational Ansatz (HEA) | 🔴 Advanced |
| 8 | Barren Plateaus | 🔴 Advanced |
| 9 | End-to-End Pipeline Demo | 🟡 Intermediate |
| 10 | Training & Evaluation | �� Intermediate |
| 11 | CaaS-Q Agent Integration | 🔴 Advanced |

---

## Chapter 1: Neural Networks & CNNs 🟢

### What is a Neural Network?

A neural network is a function $f(\mathbf{x}; \boldsymbol{\theta})$ that maps inputs to outputs via layers of learnable weights:

$$\mathbf{y} = f(\mathbf{x}) = \sigma(W_n \cdots \sigma(W_2 \cdot \sigma(W_1 \mathbf{x} + b_1) + b_2) \cdots + b_n)$$

- $W_i$ = weight matrices (the "parameters")
- $b_i$ = bias vectors
- $\sigma$ = activation function (ReLU, sigmoid, etc.)

### Convolutional Neural Networks (CNNs)

CNNs are specialized for images. Instead of fully-connected layers, they use **convolutional filters** that slide across the image detecting patterns (edges, textures, shapes):

```
Input Image → [Conv + ReLU] → [Conv + ReLU] → [Pool] → [FC] → Output
   224×224      feature maps     deeper features   compressed   class probabilities
```

### ResNet-18 (Our Backbone)

ResNet-18 is a CNN with 18 layers and **skip connections** that prevent gradient vanishing. It was pre-trained on ImageNet (1.2M images, 1000 classes).

**In MedQCNN:** We "freeze" ResNet — its weights are locked. It acts as a pure feature extractor, converting images to 512-dimensional vectors.

In [ ]:
# ===== SETUP =====
import sys
from pathlib import Path

project_root = str(Path.cwd().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import matplotlib.pyplot as plt
import numpy as np
import torch

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Chapter 1 Demo: How a CNN processes an image
from medqcnn.classical.backbone import ClassicalBackbone

backbone = ClassicalBackbone(pretrained=False, freeze=True)

# A fake grayscale medical image
image = torch.randn(1, 1, 224, 224)
print(f"Input image shape:  {list(image.shape)}  (batch=1, channels=1, 224×224)")

with torch.no_grad():
    features = backbone(image)
print(f"Output features:    {list(features.shape)}  (batch=1, 512 features)")
compression = 224 * 224 / features.shape[1]
print(
    f"\nCompression ratio: {224*224} pixels"
    f" → {features.shape[1]} features"
    f" = {compression:.0f}× compression"
)
print(f"\nResNet Parameters: {sum(p.numel() for p in backbone.parameters()):,}")
print("  (All frozen — we don't train these)")

## Chapter 2: Medical Image Classification 🟢

### The Problem

Medical imaging datasets are **small** and **expensive**:

| Dataset | ImageNet | BreastMNIST |
|---------|----------|-------------|
| Samples | 1,200,000 | 780 |
| Classes | 1,000 | 2 |
| Image size | 224×224 | 28×28 |
| Labels by | Amazon Turk | Radiologists |
| Cost per label | $0.01 | $50+ |

With only ~780 samples and a model with millions of parameters, classical networks **overfit** — they memorize training data instead of learning generalizable patterns.

### Our Approach

1. **Freeze** a pre-trained backbone (reuse ImageNet knowledge)
2. **Compress** to a small latent vector ($\mathbb{R}^{256}$)
3. Use a **quantum circuit** as the classifier (far fewer params)

In [ ]:
# Chapter 2 Demo: Load real medical data
from medqcnn.data.loader import get_medmnist_loaders

train_loader, val_loader, test_loader = get_medmnist_loaders(
    'breastmnist', batch_size=8, download=True
)

print("BreastMNIST Dataset Statistics")
print("=" * 40)
print(f"  Train samples: {len(train_loader.dataset)}")
print(f"  Val samples:   {len(val_loader.dataset)}")
print(f"  Test samples:  {len(test_loader.dataset)}")

# Visualize
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i in range(8):
    ax = axes[i // 4][i % 4]
    ax.imshow(images[i].squeeze(), cmap='gray')
    label = "Malignant" if labels[i].item() == 1 else "Benign"
    color = "red" if labels[i].item() == 1 else "green"
    ax.set_title(label, color=color, fontweight='bold')
    ax.axis('off')
plt.suptitle("BreastMNIST: Breast Ultrasound Samples", fontsize=14)
plt.tight_layout()
plt.show()

## Chapter 3: Quantum Computing Basics 🟡

### Classical Bits vs Qubits

A classical bit is either 0 or 1. A **qubit** can be in a **superposition** of both:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$$

where $|\alpha|^2 + |\beta|^2 = 1$ (probabilities must sum to 1).

### The Bloch Sphere

A qubit's state can be visualized as a point on a sphere:
- North pole = $|0\rangle$
- South pole = $|1\rangle$  
- Equator = equal superposition

### Multiple Qubits = Exponential States

| Qubits | Basis States | Classical Equivalent |
|--------|-------------|---------------------|
| 1 | 2 ($|0\rangle, |1\rangle$) | 1 bit |
| 2 | 4 ($|00\rangle, |01\rangle, |10\rangle, |11\rangle$) | 2 bits |
| 4 | 16 | 4 bits |
| 8 | 256 | 8 bits |
| $n$ | $2^n$ | $n$ bits |

**Key insight:** With 8 qubits, we can represent a vector with 256 complex amplitudes — this is the **Hilbert space** $\mathcal{H}_{2^8}$.

In [ ]:
# Chapter 3 Demo: Qubits in PennyLane
import pennylane as qml

# Create a simple quantum device
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def qubit_demo(theta):
    qml.RY(theta, wires=0)  # Rotate around Y axis
    return qml.expval(qml.PauliZ(0))

# Sweep theta from 0 to 2π
angles = np.linspace(0, 2*np.pi, 50)
expectations = [float(qubit_demo(a)) for a in angles]

plt.figure(figsize=(10, 4))
plt.plot(angles, expectations, 'b-', linewidth=2)
plt.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('Rotation Angle θ (radians)')
plt.ylabel('⟨σ_z⟩ Expectation Value')
plt.title('Single Qubit: Ry(θ) rotation → Pauli-Z measurement')
plt.xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi],
           ['0', 'π/2', 'π', '3π/2', '2π'])
plt.grid(True, alpha=0.3)
plt.fill_between(angles, expectations, alpha=0.1, color='blue')
plt.annotate('|0⟩ state\n⟨σ_z⟩ = +1', xy=(0, 1), fontsize=10, ha='left')
plt.annotate('|1⟩ state\n⟨σ_z⟩ = -1', xy=(np.pi, -1), fontsize=10, ha='center')
plt.tight_layout()
plt.show()

print("When θ=0:    qubit is in |0⟩, measurement = +1")
print("When θ=π:    qubit is in |1⟩, measurement = -1")
print("When θ=π/2:  qubit is in superposition, measurement ≈ 0")

## Chapter 4: Quantum Gates & Circuits 🟡

### Single-Qubit Gates

| Gate | Matrix | Effect |
|------|--------|--------|
| $R_y(\theta)$ | Rotation around Y | Mixes $|0\rangle \leftrightarrow |1\rangle$ |
| $R_z(\theta)$ | Rotation around Z | Adds relative phase |
| $X$ (NOT) | Bit flip | $|0\rangle \rightarrow |1\rangle$ |
| $H$ (Hadamard) | Creates superposition | $|0\rangle \rightarrow \frac{|0\rangle + |1\rangle}{\sqrt{2}}$ |

### Two-Qubit Gates: Entanglement

The **Controlled-Z (CZ)** gate entangles two qubits:

$$CZ|11\rangle = -|11\rangle, \quad CZ|\text{other}\rangle = |\text{other}\rangle$$

Entangled qubits are **correlated** — measuring one instantly affects the other. This creates non-local correlations that classical systems can't efficiently simulate.

### Building a Circuit

In MedQCNN, each ansatz layer is:
```
Qubit 0: ─Ry(θ₀)─Rz(θ₁)─●───────
Qubit 1: ─Ry(θ₂)─Rz(θ₃)─Z──●────
Qubit 2: ─Ry(θ₄)─Rz(θ₅)────Z──●─
Qubit 3: ─Ry(θ₆)─Rz(θ₇)───────Z─  (ring: Q3→Q0 too)
```

In [ ]:
# Chapter 4 Demo: Visualize entanglement
dev2 = qml.device("default.qubit", wires=2)

@qml.qnode(dev2)
def entanglement_demo(theta):
    qml.RY(theta, wires=0)
    qml.CNOT(wires=[0, 1])
    return [qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1))]

angles = np.linspace(0, 2*np.pi, 50)
z0_vals, z1_vals = [], []
for a in angles:
    r = entanglement_demo(a)
    z0_vals.append(float(r[0]))
    z1_vals.append(float(r[1]))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(angles, z0_vals, 'b-', linewidth=2, label='Qubit 0')
ax.plot(angles, z1_vals, 'r--', linewidth=2, label='Qubit 1')
ax.set_xlabel('Rotation Angle θ')
ax.set_ylabel('⟨σ_z⟩')
ax.set_title('Entangled Qubits: Rotating Q0 affects Q1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice: Q0 and Q1 are perfectly correlated!")
print("This is entanglement — the basis of quantum advantage.")

## Chapter 5: The Hybrid Architecture 🟡

### Why Hybrid?

Neither classical nor quantum alone is optimal:

| Approach | Strengths | Weaknesses |
|----------|-----------|------------|
| Classical only | Great at images, scalable | Too many parameters, overfits |
| Quantum only | Few parameters, expressive | Can't process raw images |
| **Hybrid** | **Best of both** | Requires careful design |

### MedQCNN Pipeline

```
Image (224×224) ──→ Node A (Classical) ──→ Node B (Quantum) ──→ Diagnosis
                   ResNet + Projector       Quantum Circuit       Classifier
                   512 → 256 features       256 amplitudes        2 classes
                   ~270K params (frozen)     64 params             ~200 params
```

### The L2 Normalization Bridge

The key connection between classical and quantum is **L2 normalization**:

$$\mathbf{z}_{\text{norm}} = \frac{\mathbf{z}}{\|\mathbf{z}\|_2}$$

This ensures $\sum_i |z_i|^2 = 1$, which is required for valid quantum state amplitudes.

In [ ]:
# Chapter 5 Demo: The full projection pipeline
from medqcnn.classical.backbone import ClassicalBackbone
from medqcnn.classical.projector import LatentProjector

N_QUBITS = 4
LATENT_DIM = 2 ** N_QUBITS  # 16

backbone = ClassicalBackbone(pretrained=False, freeze=True)
projector = LatentProjector(input_dim=backbone.feature_dim, latent_dim=LATENT_DIM)

image = torch.randn(1, 1, 224, 224)

with torch.no_grad():
    features = backbone(image)    # 224×224 → 512
    z = projector(features)       # 512 → 16, L2 normalized

print("Classical Compression Pipeline")
print("=" * 50)
print(f"  Image:       {list(image.shape)} = {224*224:,} values")
print(f"  ResNet out:  {list(features.shape)} = {features.shape[1]} features")
print(f"  Projected:   {list(z.shape)} = {z.shape[1]} features")
print(f"  L2 norm:     {torch.norm(z).item():.6f} (should be ≈ 1.0)")
print(
    f"\n  Total compression: {224*224:,}"
    f" → {LATENT_DIM} = {224*224/LATENT_DIM:.0f}× reduction"
)

# Visualize the latent vector
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(range(LATENT_DIM), z[0].detach().numpy(), color='steelblue')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Latent Vector z ∈ R^{LATENT_DIM} (L2 normalized)')
axes[0].grid(True, alpha=0.2)

axes[1].bar(range(LATENT_DIM), z[0].detach().numpy()**2, color='orange')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('|z_i|²')
axes[1].set_title(f'Probability Distribution (sum = {(z[0]**2).sum():.4f})')
axes[1].grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Chapter 6: Amplitude Encoding 🔴

### The Mathematical Bridge: Classical → Quantum

Given an L2-normalized vector $\mathbf{z} = (z_0, z_1, \ldots, z_{2^n-1})$, amplitude encoding creates:

$$|\psi(\mathbf{z})\rangle = \sum_{i=0}^{2^n - 1} z_i |i\rangle$$

For 4 qubits ($2^4 = 16$ basis states):

$$|\psi\rangle = z_0|0000\rangle + z_1|0001\rangle + z_2|0010\rangle + \cdots + z_{15}|1111\rangle$$

### Why This Works

Each $z_i$ becomes the **probability amplitude** of measuring the system in basis state $|i\rangle$:
- Probability of measuring $|i\rangle$ = $|z_i|^2$
- Total probability: $\sum |z_i|^2 = \|\mathbf{z}\|_2^2 = 1$ ✓

### Exponential Compression

| Method | Classical Values Encoded | Qubits Required |
|--------|------------------------|-----------------|
| **Basis encoding** | $n$ values | $n$ qubits |
| **Amplitude encoding** | $2^n$ values | $n$ qubits |

With amplitude encoding, 8 qubits encode **256 features**. Basis encoding would need 256 qubits!

In [ ]:
# Chapter 6 Demo: Amplitude encoding in action
from medqcnn.quantum.encoding import amplitude_encode

dev4 = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev4)
def encoding_demo(features):
    amplitude_encode(features, wires=range(N_QUBITS))
    return qml.state()

# Encode our projected latent vector
z_np = z[0].detach().numpy().astype(float)
z_np = z_np / np.linalg.norm(z_np)

state = encoding_demo(z_np)

print("Amplitude Encoding Result")
print("=" * 50)
print(f"  Input: {len(z_np)} classical values → {N_QUBITS} qubits")
print(f"  Quantum state has {len(state)} amplitudes")
print("\n  First 8 amplitudes:")
for i in range(8):
    bar = "█" * int(abs(state[i].real) * 30)
    print(f"    |{i:04b}⟩ = {state[i].real:+.4f}  {bar}")

# Verify: quantum amplitudes should match input
match = np.allclose(state.real[:LATENT_DIM], z_np, atol=1e-5)
print(f"\n  Input matches quantum state: {'✓ Yes' if match else '✗ No'}")

## Chapter 7: Variational Ansatz (HEA) 🔴

### What is an Ansatz?

An **ansatz** is a parameterized quantum circuit — a circuit whose gate angles $\boldsymbol{\theta}$ the optimizer can tune. It's the quantum equivalent of a neural network layer.

### Hardware-Efficient Ansatz (HEA)

MedQCNN uses the HEA pattern, designed to run on real quantum hardware:

**Per layer:**
1. Apply $R_y(\theta_{i,0})$ and $R_z(\theta_{i,1})$ to each qubit
2. Apply $CZ$ gates between neighbors (ring topology: 0↔1, 1↔2, ..., (n-1)↔0)

**Parameters per layer:** $2 \times n_{\text{qubits}}$ (one $R_y$ and one $R_z$ per qubit)

### Why Ring Entanglement?

The ring topology $(Q_0-Q_1-Q_2-\cdots-Q_{n-1}-Q_0)$ ensures:
- Every qubit is entangled with at least 2 neighbors
- Information can flow across all qubits in $\lceil n/2 \rceil$ layers
- Minimal gate depth (important for noisy hardware)

In [ ]:
# Chapter 7 Demo: Visualize the full quantum circuit
from pennylane import numpy as pnp

from medqcnn.quantum.ansatz import get_param_shape
from medqcnn.quantum.qnode import create_qnode

qnode = create_qnode(n_qubits=N_QUBITS, n_layers=2)

sample_features = np.random.rand(LATENT_DIM)
sample_features = sample_features / np.linalg.norm(sample_features)
sample_params = pnp.random.randn(2, N_QUBITS, 2) * 0.1

# Draw circuit
print("Full Quantum Circuit (2 layers, 4 qubits)")
print("=" * 60)
fig, ax = qml.draw_mpl(qnode)(sample_features, sample_params)
plt.tight_layout()
plt.show()

# Parameter count analysis
from medqcnn.config.constants import NUM_ANSATZ_LAYERS

shape = get_param_shape(n_layers=NUM_ANSATZ_LAYERS, n_qubits=N_QUBITS)
total = int(np.prod(shape))

print(
    f"\nParameter Analysis"
    f" (Production: {NUM_ANSATZ_LAYERS} layers,"
    f" {N_QUBITS} qubits)"
)
print(
    f"  Shape: {shape} = {shape[0]} layers"
    f" × {shape[1]} qubits × {shape[2]} rotations"
)
print(f"  Total quantum parameters: {total}")
classical_params = LATENT_DIM**2 + LATENT_DIM
print(
    f"\n  Equivalent classical FC layer:"
    f" {LATENT_DIM}×{LATENT_DIM} + {LATENT_DIM}"
    f" = {classical_params} params"
)
savings = classical_params - total
pct = (1 - total / classical_params) * 100
print(
    f"  Quantum saves {savings} fewer"
    f" parameters ({pct:.0f}% reduction)"
)


## Chapter 8: The Barren Plateau Problem 🔴

### The Crisis

When training variational quantum circuits, there's a fundamental mathematical problem. For a random parameterized circuit, the gradient of the loss function **vanishes exponentially** with qubit count:

$$\text{Var}\left(\frac{\partial \mathcal{L}}{\partial \theta_k}\right) \leq F(n) \cdot 2^{-n}$$

For 20 qubits: $2^{-20} \approx 0.000001$ — the optimizer sees a flat landscape.

### Three Mitigation Strategies in MedQCNN

**1. Local Cost Functions** — We measure each qubit individually ($\langle\sigma_z^{(i)}\rangle$) rather than a global fidelity. Local costs have **polynomial** (not exponential) gradient variance.

**2. Shallow Circuits** — Only 4 ansatz layers. Gradient variance is bounded by $O(\text{poly}(n) \cdot 2^{-n/L})$ where $L$ is the "light cone" depth. Fewer layers = larger gradients.

**3. Small Qubit Count** — 8 qubits max. Even with worst-case $2^{-8} = 0.004$, gradients are still trainable. At $n=20$, they wouldn't be.

In [ ]:
# Chapter 8 Demo: Gradient variance vs qubit count
# Simulate the barren plateau effect
np.random.seed(42)

qubit_counts = [2, 3, 4, 5, 6, 7, 8, 10, 12]
grad_variances_global = []
grad_variances_local = []

for n in qubit_counts:
    # Theoretical bounds
    global_var = 2**(-n)        # Global cost: exponential decay
    local_var = 2**(-n/3)       # Local cost: much slower decay
    grad_variances_global.append(global_var)
    grad_variances_local.append(local_var)

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(
    qubit_counts, grad_variances_global,
    'ro-', linewidth=2, markersize=8,
    label='Global cost (BAD)',
)
ax.semilogy(
    qubit_counts, grad_variances_local,
    'go-', linewidth=2, markersize=8,
    label='Local cost (MedQCNN)',
)
ax.axvline(x=8, color='blue', linestyle='--', alpha=0.7, label='MedQCNN max (8 qubits)')
ax.axhline(y=1e-4, color='gray', linestyle=':', alpha=0.5)
ax.annotate('Trainability threshold', xy=(3, 1.5e-4), fontsize=9, color='gray')
ax.set_xlabel('Number of Qubits')
ax.set_ylabel('Gradient Variance (log scale)')
ax.set_title('Barren Plateau: Global vs Local Cost Functions')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print("At 8 qubits:")
print(f"  Global cost gradient variance: {2**(-8):.6f} (barely trainable)")
print(f"  Local cost gradient variance:  {2**(-8/3):.6f} (MedQCNN — trainable!)")

## Chapter 9: End-to-End Pipeline Demo 🟡

Now let's assemble ALL components and run a complete inference.

In [ ]:
# Chapter 9: Build the complete HybridQCNN
from medqcnn.config.constants import NUM_ANSATZ_LAYERS
from medqcnn.model.hybrid import HybridQCNN

model = HybridQCNN(
    n_qubits=N_QUBITS,
    n_layers=NUM_ANSATZ_LAYERS,
    n_classes=2,
    pretrained=False,
)

# Parameter breakdown
params = model.count_trainable_params()
print("HybridQCNN Parameter Breakdown")
print("=" * 50)

categories = ['projector', 'quantum', 'classifier']
values = [params[c] for c in categories]
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(categories, values, color=colors, edgecolor='white', linewidth=2)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Parameters')
axes[0].set_title('Trainable Parameters by Component')
axes[0].grid(True, alpha=0.2, axis='y')

# Pie chart
axes[1].pie(values, labels=categories, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title(f'Total: {params["total"]:,} trainable parameters')
plt.tight_layout()
plt.show()

print(f"\nQuantum params: only {params['quantum']} — yet they operate")
print(f"in a {LATENT_DIM}-dimensional Hilbert space!")

In [ ]:
# Forward pass with step-by-step visualization
model.eval()
test_image = torch.randn(1, 1, 224, 224)

with torch.no_grad():
    # Step-by-step
    features = model.backbone(test_image)
    z = model.projector(features)
    q_out = model.quantum_layer(z)
    logits = model.classifier(q_out)
    probs = torch.softmax(logits, dim=1)

print("Step-by-Step Inference")
print("=" * 50)
print(f"  1. Image          → {list(test_image.shape)}")
print(f"  2. ResNet features → {list(features.shape)}")
print(f"  3. Projected z     → {list(z.shape)} (L2 norm = {torch.norm(z):.4f})")
print(f"  4. Quantum output  → {list(q_out.shape)} (per-qubit ⟨σ_z⟩)")
print(f"     Values: {[f'{v:.4f}' for v in q_out[0].tolist()]}")
print(f"  5. Logits          → {logits[0].tolist()}")
print(f"  6. Probabilities   → Benign: {probs[0,0]:.4f}, Malignant: {probs[0,1]:.4f}")
print(
    f"  7. Prediction      →"
    f" {'Malignant' if logits.argmax().item() == 1 else 'Benign'}"
)


## Chapter 10: Training & Evaluation 🟡

### The Training Loop

1. **Forward pass:** Image → HybridQCNN → logits
2. **Loss:** Cross-entropy + L2 regularization on quantum parameters
3. **Backward pass:** PyTorch autograd computes gradients through the quantum circuit (via `backprop` or `parameter-shift`)
4. **Update:** Adam optimizer adjusts projector, quantum, and classifier weights

### Key: Gradient Flow Through Quantum

PennyLane's `TorchLayer` makes the quantum circuit a native PyTorch module. Gradients flow seamlessly:

```
Loss → ∂L/∂classifier → ∂L/∂quantum_params → ∂L/∂projector
```

In [ ]:
# Chapter 10 Demo: Quick training run
from medqcnn.training.trainer import Trainer

model = HybridQCNN(
    n_qubits=N_QUBITS,
    n_layers=NUM_ANSATZ_LAYERS,
    n_classes=2,
    pretrained=False,
)

# Resize MedMNIST 28×28 → 224×224 for ResNet
class ResizedLoader:
    def __init__(self, loader):
        self.loader = loader
        self.dataset = loader.dataset
    def __iter__(self):
        for imgs, lbls in self.loader:
            imgs = torch.nn.functional.interpolate(
                imgs.float(), size=(224, 224), mode='bilinear', align_corners=False
            )
            yield imgs, lbls
    def __len__(self):
        return len(self.loader)

trainer = Trainer(
    model=model,
    train_loader=ResizedLoader(train_loader),
    val_loader=ResizedLoader(val_loader),
    learning_rate=1e-3,
    device='cpu',
)

print("Training HybridQCNN for 3 epochs...")
print("(This may take a few minutes on CPU)\n")
history = trainer.fit(epochs=3, save_every=3)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train', markersize=8)
axes[0].plot(epochs, history['val_loss'], 'r-o', label='Val', markersize=8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train', markersize=8)
axes[1].plot(epochs, history['val_acc'], 'r-o', label='Val', markersize=8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

plt.suptitle('MedQCNN Training Progress', fontsize=14)
plt.tight_layout()
plt.show()

## Chapter 11: CaaS-Q Agent Integration 🔴

### MedQCNN as a Tool

In the CaaS-Q platform, MedQCNN is not an app — it's a **tool** that AI agents call:

```
Doctor → Agent (LLM) → quantum_diagnose tool → Clinical Report
```

### Integration Options

| Method | Protocol | Best For |
|--------|----------|----------|
| **LangChain `@tool`** | Python | Direct agent integration |
| **MCP Server** | stdio | Claude, Copilot, IDE agents |
| **REST API** | HTTP | Docker microservices |
| **Kafka** | Event-driven | High-throughput pipelines |

### The Agent Loop

1. Doctor uploads image
2. LLM agent decides to call `quantum_diagnose`
3. MedQCNN runs quantum inference
4. Agent interprets quantum expectation values
5. Agent writes clinical report with disclaimers

In [ ]:
# Chapter 11 Demo: Using MedQCNN as a LangChain tool
import json

from medqcnn.agent.tools import get_model_info, list_medical_datasets

# Tool 1: What datasets are available?
datasets = json.loads(list_medical_datasets.invoke(""))
print("Available Medical Datasets")
print("=" * 50)
for name, ds in datasets.items():
    print(f"  {name:15s} — {ds['description']}")

# Tool 2: Model info
info = json.loads(get_model_info.invoke(""))
print(f"\nModel: {info['model']}")
print(f"Quantum advantage: {info['quantum_advantage']}")

In [ ]:
# Tool 3: Run a diagnosis
import numpy as np
from PIL import Image

from medqcnn.agent.agent import run_diagnostic_without_llm

# Save a test image
images, labels = next(iter(test_loader))
img_array = (images[0].squeeze().numpy() * 255).astype(np.uint8)
save_path = Path("../data/test_sample.png")
save_path.parent.mkdir(exist_ok=True)
Image.fromarray(img_array).save(save_path)

# Generate clinical report (no LLM needed)
report = run_diagnostic_without_llm(str(save_path.resolve()))
print(report)

---

## 🎓 What You've Learned

| Chapter | Key Takeaway |
|---------|-------------|
| 1 | CNNs extract image features via convolutional filters |
| 2 | Medical datasets are small — classical models overfit |
| 3 | Qubits use superposition to represent exponential states |
| 4 | Quantum gates (Ry, Rz, CZ) manipulate qubits |
| 5 | Hybrid = classical compression + quantum processing |
| 6 | Amplitude encoding maps vectors to quantum states |
| 7 | HEA ansatz: trainable gates with ring entanglement |
| 8 | Barren plateaus are mitigated by local cost functions |
| 9 | The full pipeline: image → ResNet → projector → quantum → diagnosis |
| 10 | Training works via PyTorch autograd through quantum circuits |
| 11 | MedQCNN is a CaaS-Q tool callable by LangChain/MCP/REST |

## 📚 Further Reading

1. `GEMINI.md` — Full project architecture and roadmap
2. `docs/quantum_primer.md` — Quantum math reference
3. `docs/api_reference.md` — REST API and MCP tool docs
4. `docs/architecture.md` — System architecture diagrams
5. `docs/deployment.md` — Docker and Pi 5 deployment guide